# 02 — Feature Engineering for Flight Delay Prediction

Build features from the raw dataset to improve delay prediction beyond the baseline (historical frequency by weekday + airport).

**Rules:**
- Target: `ArrDel15` (ADR-0001)
- Exclude cancelled flights (ADR-0002)
- Random seed: 42 for reproducibility

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load data and apply base filters

In [ ]:
df = pd.read_csv('../data/flights.csv')

# Exclude cancelled flights (ADR-0002)
df = df[df['Cancelled'] == 0].copy()

# Drop rows where target is missing
df = df.dropna(subset=['ArrDel15'])
df['ArrDel15'] = df['ArrDel15'].astype(int)

print(f'Working dataset: {len(df):,} rows')
print(f'Delay rate: {df["ArrDel15"].mean():.3f}')

## 2. Time-of-day features

Convert `CRSDepTime` (scheduled departure, HHMM format) into useful categories.

In [ ]:
def extract_hour(hhmm):
    """Convert HHMM integer to hour (0-23)."""
    if pd.isna(hhmm):
        return np.nan
    hhmm = int(hhmm)
    return hhmm // 100

df['DepHour'] = df['CRSDepTime'].apply(extract_hour)

# Time-of-day buckets
def time_bucket(hour):
    if pd.isna(hour):
        return 'unknown'
    if hour < 6:
        return 'red_eye'     # 00:00-05:59
    elif hour < 12:
        return 'morning'     # 06:00-11:59
    elif hour < 18:
        return 'afternoon'   # 12:00-17:59
    else:
        return 'evening'     # 18:00-23:59

df['TimeBucket'] = df['DepHour'].apply(time_bucket)

# Visualize delay rate by hour
by_hour = df.groupby('DepHour')['ArrDel15'].mean().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_hour['DepHour'], by_hour['ArrDel15'] * 100, color='#5B8DEF')
ax.set_xlabel('Departure Hour')
ax.set_ylabel('Delay rate (%)')
ax.set_title('Delay rate by scheduled departure hour')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

## 3. Route feature

Combine origin and destination into a single route identifier.

In [ ]:
df['Route'] = df['OriginAirportID'].astype(str) + '-' + df['DestAirportID'].astype(str)

n_routes = df['Route'].nunique()
print(f'Unique routes: {n_routes}')

# Route delay rates (for routes with enough data)
route_stats = df.groupby('Route')['ArrDel15'].agg(['mean', 'count']).reset_index()
route_stats.columns = ['Route', 'route_delay_rate', 'route_flights']
print(f'Routes with >= 50 flights: {(route_stats["route_flights"] >= 50).sum()}')

# Merge route-level delay rate as a feature (target encoding approach)
df = df.merge(route_stats[['Route', 'route_delay_rate', 'route_flights']], on='Route', how='left')

## 4. Carrier encoding

Compute carrier-level delay rate as a feature.

In [ ]:
carrier_stats = df.groupby('Carrier')['ArrDel15'].mean().reset_index()
carrier_stats.columns = ['Carrier', 'carrier_delay_rate']
df = df.merge(carrier_stats, on='Carrier', how='left')

print('Carrier delay rates:')
print(carrier_stats.sort_values('carrier_delay_rate', ascending=False).to_string(index=False))

## 5. Airport-level features

Compute origin and destination airport delay rates.

In [ ]:
# Destination airport delay rate
dest_stats = df.groupby('DestAirportID')['ArrDel15'].mean().reset_index()
dest_stats.columns = ['DestAirportID', 'dest_delay_rate']
df = df.merge(dest_stats, on='DestAirportID', how='left')

# Origin airport delay rate
origin_stats = df.groupby('OriginAirportID')['ArrDel15'].mean().reset_index()
origin_stats.columns = ['OriginAirportID', 'origin_delay_rate']
df = df.merge(origin_stats, on='OriginAirportID', how='left')

print(f'Origin airports: {df["OriginAirportID"].nunique()}')
print(f'Dest airports:   {df["DestAirportID"].nunique()}')

## 6. Season / month features

In [ ]:
def month_to_season(m):
    if m in [12, 1, 2]:
        return 'winter'
    elif m in [3, 4, 5]:
        return 'spring'
    elif m in [6, 7, 8]:
        return 'summer'
    else:
        return 'fall'

df['Season'] = df['Month'].apply(month_to_season)

by_season = df.groupby('Season')['ArrDel15'].mean().reset_index()
print('Delay rate by season:')
print(by_season.to_string(index=False))

## 7. Final feature matrix

Assemble the feature set for modeling.

In [ ]:
# Numerical features
num_features = [
    'Month', 'DayOfWeek', 'DepHour',
    'carrier_delay_rate', 'origin_delay_rate', 'dest_delay_rate',
    'route_delay_rate', 'route_flights'
]

# Categorical features (will be one-hot encoded in modeling notebook)
cat_features = ['TimeBucket', 'Season']

target = 'ArrDel15'

# Check for missing values in features
all_features = num_features + cat_features
print('Missing values per feature:')
print(df[all_features].isnull().sum())
print(f'\nTotal rows: {len(df):,}')

In [ ]:
# Save the prepared dataset for the modeling notebook
cols_to_save = all_features + [target, 'OriginAirportID', 'DestAirportID', 'Carrier', 'Route']
df_prepared = df[cols_to_save].dropna()
df_prepared.to_csv('../data/flights_prepared.csv', index=False)
print(f'Saved prepared dataset: {len(df_prepared):,} rows, {len(cols_to_save)} columns')
df_prepared.head()

## 8. Feature correlation with target

In [ ]:
corr_with_target = df_prepared[num_features + [target]].corr()[target].drop(target).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#F44336' if v > 0 else '#4CAF50' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.set_xlabel('Correlation with ArrDel15')
ax.set_title('Feature correlation with arrival delay')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## Summary

Features engineered:

| Feature | Type | Description |
|---------|------|-------------|
| `Month` | Numeric | Month of year (1-12) |
| `DayOfWeek` | Numeric | Day of week (1=Mon, 7=Sun) |
| `DepHour` | Numeric | Scheduled departure hour (0-23) |
| `TimeBucket` | Categorical | red_eye/morning/afternoon/evening |
| `Season` | Categorical | winter/spring/summer/fall |
| `carrier_delay_rate` | Numeric | Historical delay rate for the carrier |
| `origin_delay_rate` | Numeric | Historical delay rate for origin airport |
| `dest_delay_rate` | Numeric | Historical delay rate for dest airport |
| `route_delay_rate` | Numeric | Historical delay rate for the route |
| `route_flights` | Numeric | Number of flights on this route |

The prepared dataset is saved to `../data/flights_prepared.csv` for the modeling notebook.